first Agentic Framework project!!



We're going to build a simple Agent system for generating cold sales outreach emails:
1. Agent workflow
2. Use of tools to call functions
3. Agent collaboration via Tools and Handoffs

In [1]:
from dotenv import load_dotenv
from agents import Agent, Runner, trace, function_tool , OpenAIChatCompletionsModel , set_tracing_disabled
from openai import AsyncOpenAI 
from openai.types.responses import ResponseTextDeltaEvent
from typing import Dict
# import sendgrid
import os , requests
# from sendgrid.helpers.mail import Mail, Email, To, Content
import asyncio


In [2]:
load_dotenv(override=True)

True

In [3]:
# set tracing disabled to True to disable tracing for all agents ( as iam not using openai key )

set_tracing_disabled(True)

### Step 1 : Agent Workflow

In [4]:
instructions1 = "You are a sales agent working for ComplAI, \
a company that provides a SaaS tool for ensuring SOC2 compliance and preparing for audits, powered by AI. \
You write professional, serious cold emails."

instructions2 = "You are a humorous, engaging sales agent working for ComplAI, \
a company that provides a SaaS tool for ensuring SOC2 compliance and preparing for audits, powered by AI. \
You write witty, engaging cold emails that are likely to get a response."

instructions3 = "You are a busy sales agent working for ComplAI, \
a company that provides a SaaS tool for ensuring SOC2 compliance and preparing for audits, powered by AI. \
You write concise, to the point cold emails."

In [5]:
openai = AsyncOpenAI(base_url = os.getenv("GITHUB_BASE_URL"),api_key=os.getenv("GITHUB_TOKEN"))  # openai async client 

model = OpenAIChatCompletionsModel(openai_client=openai, model="gpt-4o-mini")  

sales_agent1 = Agent(
        name="Professional Sales Agent",
        instructions=instructions1,
        model=model
)

sales_agent2 = Agent(
        name="Engaging Sales Agent",
        instructions=instructions2,
        model=model
)

sales_agent3 = Agent(
        name="Busy Sales Agent",
        instructions=instructions3,
        model=model
)

In [6]:

result = Runner.run_streamed(sales_agent1, input="Write a cold sales email")
async for event in result.stream_events():
    if event.type == "raw_response_event" and isinstance(event.data, ResponseTextDeltaEvent):
        print(event.data.delta, end="", flush=True)

Subject: Simplify Your SOC2 Compliance Journey with ComplAI

Hi [Recipient's Name],

I hope this message finds you well. My name is [Your Name], and I’m reaching out to introduce you to ComplAI, a cutting-edge SaaS solution designed to streamline your SOC2 compliance process and enhance your audit preparedness.

In today's regulatory environment, maintaining compliance can be a daunting task. Our AI-powered platform automates critical compliance workflows, enabling your team to focus on what matters most—growing your business. With ComplAI, you can expect:

- **Automated Documentation**: Generate and manage all necessary compliance documents effortlessly.
- **Real-Time Monitoring**: Keep track of your compliance status with AI-driven insights.
- **Audit Preparedness**: Ensure you’re always ready for your next audit with organized records and audits trails.

Many companies like [Example of a similar company or industry] have successfully reduced their compliance time by up to 50%, allow

In [7]:
message = "Write a cold sales email"

# with trace("Parallel cold emails"):
#     results = await asyncio.gather(
#         Runner.run(sales_agent1, message),
#         Runner.run(sales_agent2, message),
#         Runner.run(sales_agent3, message),
#     )


results = await asyncio.gather(
    Runner.run(sales_agent1, message),
    Runner.run(sales_agent2, message),
    Runner.run(sales_agent3, message),
)

outputs = [result.final_output for result in results]

for output in outputs:
    print(output + "\n\n")


Subject: Elevate Your SOC2 Compliance Efforts with ComplAI

Dear [Recipient's Name],

I hope this message finds you well. My name is [Your Name], and I am with ComplAI, where we specialize in simplifying SOC2 compliance and preparation for audits through our advanced AI-driven SaaS platform.

As your organization navigates the complexities of compliance, I wanted to share how ComplAI can help streamline your processes, reduce manual effort, and enhance your audit preparedness. Our platform not only automates compliance tasks but also provides real-time insights and tracking, allowing you to focus on your core business objectives without the burden of regulatory concerns.

Key benefits of ComplAI include:

- Automated evidence collection and reporting
- Real-time monitoring of compliance status
- Reduced time spent on audits and assessments
- Customized compliance workflows tailored to your needs

We understand that maintaining compliance can be challenging, especially in today’s fast-p

In [8]:
sales_picker = Agent(
    name="sales_picker",
    instructions="You pick the best cold sales email from the given options. \
Imagine you are a customer and pick the one you are most likely to respond to. \
Do not give an explanation; reply with the selected email only.",
    model=model
)

In [9]:
message = "Write the cold sales email"

results = await asyncio.gather(
        Runner.run(sales_agent1, message),
        Runner.run(sales_agent2, message),
        Runner.run(sales_agent3, message),
    )
outputs = [result.final_output for result in results]

emails = "Cold sales emails:\n\n" + "\n\nEmail:\n\n".join(outputs)

best = await Runner.run(sales_picker, emails)

print(f"Best sales email:\n{best.final_output}")

Best sales email:
Subject: Unlock the Secrets of SOC2 Compliance – No Spellbook Required! 🧙‍♂️✨

Hi [Recipient's Name],

Do you ever feel like navigating SOC2 compliance is more complex than assembling IKEA furniture without the instruction manual? We’ve been there! 

At ComplAI, we believe compliance should be as easy as pie (or at least easier than explaining what "SOC2" actually stands for). Our SaaS tool harnesses the power of AI to help you breeze through compliance challenges and get audited with the confidence of a cat walking on a ledge (but way less chaotic).

Here’s what you can expect:
- A streamlined process that makes compliance feel like a walk in the park (maybe even a stroll through a cat cafe).
- Automated documentation prep, so you can spend less time stressing and more time mastering that elusive work-life balance.
- Built-in AI that’s smarter than your average bear, making sure you’re always on top of the latest requirements. 

Curious to learn more? Let’s schedule 

## Part 2 : use of tools 

In [10]:
sales_agent1 = Agent(
        name="Professional Sales Agent",
        instructions=instructions1,
        model=model,
)

sales_agent2 = Agent(
        name="Engaging Sales Agent",
        instructions=instructions2,
        model=model,
)

sales_agent3 = Agent(
        name="Busy Sales Agent",
        instructions=instructions3,
        model=model,
)

In [11]:
sales_agent1

Agent(name='Professional Sales Agent', handoff_description=None, tools=[], mcp_servers=[], mcp_config={}, instructions='You are a sales agent working for ComplAI, a company that provides a SaaS tool for ensuring SOC2 compliance and preparing for audits, powered by AI. You write professional, serious cold emails.', prompt=None, handoffs=[], model=<agents.models.openai_chatcompletions.OpenAIChatCompletionsModel object at 0x0000025C543EFBF0>, model_settings=ModelSettings(temperature=None, top_p=None, frequency_penalty=None, presence_penalty=None, tool_choice=None, parallel_tool_calls=None, truncation=None, max_tokens=None, reasoning=None, verbosity=None, metadata=None, store=None, prompt_cache_retention=None, include_usage=None, response_include=None, top_logprobs=None, extra_query=None, extra_body=None, extra_headers=None, extra_args=None), input_guardrails=[], output_guardrails=[], output_type=None, hooks=None, tool_use_behavior='run_llm_again', reset_tool_choice=True)

### Steps 2 and 3 : Tools and Agent interactions 

In [12]:
# and we can also convert an agent into a tool 
tool1 = sales_agent1.as_tool(tool_name="sales_agent1", tool_description="Write a cold sales email")
tool1

FunctionTool(name='sales_agent1', description='Write a cold sales email', params_json_schema={'description': 'Default input schema for agent-as-tool calls.', 'properties': {'input': {'title': 'Input', 'type': 'string'}}, 'required': ['input'], 'title': 'AgentAsToolInput', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<function Agent.as_tool.<locals>._run_agent_tool at 0x0000025C5515ED40>, strict_json_schema=True, is_enabled=True, tool_input_guardrails=None, tool_output_guardrails=None, needs_approval=False)

In [13]:



@function_tool
def send_email(body: str) -> dict[str, str]:
    import resend

    resend.api_key = os.getenv("RESEND_API_KEY")

    params: resend.Emails.SendParams = {
    "from": "onboarding@resend.dev",
    "to": os.getenv("TO_EMAIL"),
    "subject": "hello world",
    "text": body,
    }

    email = resend.Emails.send(params)
    print(email)
    return {"status": "success"}


# send_email("Hello, this is a test email sent from the send_email function.")

#### so now we gather all tools together 
A tool for each of our 3 email writing agents 
And  a tool for our function to send emails 

In [14]:
description = "Write a cold sales email"

tool1 = sales_agent1.as_tool(tool_name="sales_agent1", tool_description=description)
tool2 = sales_agent2.as_tool(tool_name="sales_agent2", tool_description=description)
tool3 = sales_agent3.as_tool(tool_name="sales_agent3", tool_description=description)

tools = [tool1, tool2, tool3,send_email]

tools

[FunctionTool(name='sales_agent1', description='Write a cold sales email', params_json_schema={'description': 'Default input schema for agent-as-tool calls.', 'properties': {'input': {'title': 'Input', 'type': 'string'}}, 'required': ['input'], 'title': 'AgentAsToolInput', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<function Agent.as_tool.<locals>._run_agent_tool at 0x0000025C54F39620>, strict_json_schema=True, is_enabled=True, tool_input_guardrails=None, tool_output_guardrails=None, needs_approval=False),
 FunctionTool(name='sales_agent2', description='Write a cold sales email', params_json_schema={'description': 'Default input schema for agent-as-tool calls.', 'properties': {'input': {'title': 'Input', 'type': 'string'}}, 'required': ['input'], 'title': 'AgentAsToolInput', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<function Agent.as_tool.<locals>._run_agent_tool at 0x0000025C54F62840>, strict_json_schema=True, is_enabled=True, tool_input_g

### And now it's time for our Sales Manager - our planning agent

In [15]:


instructions = """
You are a Sales Manager at ComplAI. Your goal is to find the single best cold sales email using the sales_agent tools.
 
Follow these steps carefully:
1. Generate Drafts: Use all three sales_agent tools to generate three different email drafts. Do not proceed until all three drafts are ready.
 
2. Evaluate and Select: Review the drafts and choose the single best email using your judgment of which one is most effective.
 
3. Use the send_email tool to send the best email (and only the best email) to the user.
 
Crucial Rules:
- You must use the sales agent tools to generate the drafts — do not write them yourself.
- You must send ONE email using the send_email tool — never more than one.
"""


sales_manager = Agent(name="Sales Manager", instructions=instructions, tools=tools, model=model)

message = "Send a cold sales email addressed to 'Dear CEO'"

result = await Runner.run(sales_manager, message)

{'id': '3564ea68-14b9-494d-9d59-b52896d2c211', 'http_headers': {'Date': 'Wed, 10 Jun 2026 17:03:07 GMT', 'Content-Type': 'application/json', 'Content-Length': '45', 'Connection': 'keep-alive', 'ratelimit-limit': '5', 'ratelimit-policy': '5;w=1', 'ratelimit-remaining': '4', 'ratelimit-reset': '1', 'x-resend-daily-quota': '1', 'x-resend-monthly-quota': '7', 'cf-cache-status': 'DYNAMIC', 'Server': 'cloudflare', 'CF-RAY': 'a099f293cfddd3a5-FRA'}}


In [ ]:
print(result.final_output)

### Handoffs represent a way an agent can delegate to an agent passing control to it 
Handoffs and Agents-as-tools are similar:

In both cases, an Agent can collaborate with another Agent

With tools, control passes back

With handoffs, control passes across


Tool is considereded as request is passed  to tool and then tool returns response    
But in handoff instead of response handoff agent can passes(delegation) control accross other agent

In [ ]:

subject_instructions = "You can write a subject for a cold sales email. \
You are given a message and you need to write a subject for an email that is likely to get a response."

html_instructions = "You can convert a text email body to an HTML email body. \
You are given a text email body which might have some markdown \
and you need to convert it to an HTML email body with simple, clear, compelling layout and design."

subject_writer = Agent(name="Email subject writer", instructions=subject_instructions, model=model)
subject_tool = subject_writer.as_tool(tool_name="subject_writer", tool_description="Write a subject for a cold sales email")

html_converter = Agent(name="HTML email body converter", instructions=html_instructions, model=model)

html_tool = html_converter.as_tool(tool_name="html_converter",tool_description="Convert a text email body to an HTML email body")


In [ ]:
@function_tool
def send_html_email(subject:str,html_body: str) -> Dict[str, str]:
    import resend

    resend.api_key = os.getenv("RESEND_API_KEY")

    params: resend.Emails.SendParams = {
    "from": "onboarding@resend.dev",
    "to": os.getenv("TO_EMAIL"),
    "subject": subject,
    "html": html_body,
    }

    email = resend.Emails.send(params)
    print(email)
    return {"status": "success"}


In [ ]:
tools = [subject_tool, html_tool, send_html_email]
tools

In [ ]:
instructions ="You are an email formatter and sender. You receive the body of an email to be sent. \
You first use the subject_writer tool to write a subject for the email, then use the html_converter tool to convert the body to HTML. \
Finally, you use the send_html_email tool to send the email with the subject and HTML body."


emailer_agent = Agent(
    name="Email Manager",
    instructions=instructions,
    tools=tools,
    model=model,
    handoff_description="Convert an email to HTML and send it")


In [ ]:
tools = [tool1, tool2, tool3]
handoffs = [emailer_agent]
print(tools)
print(handoffs)

In [ ]:
# Improved instructions thanks to student Guillermo F.

sales_manager_instructions = """
You are a Sales Manager at ComplAI. Your goal is to find the single best cold sales email using the sales_agent tools.
 
Follow these steps carefully:
1. Generate Drafts: Use all three sales_agent tools to generate three different email drafts. Do not proceed until all three drafts are ready.
 
2. Evaluate and Select: Review the drafts and choose the single best email using your judgment of which one is most effective.
You can use the tools multiple times if you're not satisfied with the results from the first try.
 
3. Handoff for Sending: Pass ONLY the winning email draft to the 'Email Manager' agent. The Email Manager will take care of formatting and sending.
 
Crucial Rules:
- You must use the sales agent tools to generate the drafts — do not write them yourself.
- You must hand off exactly ONE email to the Email Manager — never more than one.
"""


sales_manager = Agent(
    name="Sales Manager",
    instructions=sales_manager_instructions,
    tools=tools,
    handoffs=handoffs,
    model=model)

message = "Send out a cold sales email addressed to Dear CEO from Aakash"

# with trace("Automated SDR"):
#     result = await Runner.run(sales_manager, message)

result = await Runner.run(sales_manager, message)

## Exercise 

- identify the Agentic design patterns that were used here?
- What is the 1 line that changed this from being an Agentic "workflow" to "agent" under Anthropic's definition?

- Try adding in more tools and Agents! You could have tools that handle the mail merge to send to a list.

- HARD CHALLENGE: research how you can have SendGrid call a Callback webhook when a user replies to an email,
Then have the SDR respond to keep the conversation going! This may require some "vibe coding"